In [4]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import math

In [6]:
debts = {
    "credit_cards": [
        {
            "name": "SBI Credit Card",
            "balance": 50000,
            "apr": 42,
            "billing_day": 1,      # Statement generated on 1st
            "due_day": 18,         # Payment due on 18th
            "min_percent": 5,      # 5% minimum payment
            "grace_days": 18       # No interest if paid by due date
        }
    ],
    "loans": [
        {
            "name": "HDFC Home Loan",
            "principal": 2500000,
            "apr": 8.5,
            "tenure_months": 240,   # 20 years
            "emi_day": 5,
            "type": "reducing"
        },
        {
            "name": "Axis Car Loan",
            "principal": 600000,
            "apr": 9.5,
            "tenure_months": 60,    # 5 years
            "emi_day": 10,
            "type": "reducing"
        },
        {
            "name": "Education Loan",
            "principal": 400000,
            "apr": 10.5,
            "tenure_months": 84,    # 7 years
            "emi_day": 15,
            "type": "reducing",
            "moratorium_until": "2026-06-30"  # Interest-only period
        }
    ]
}


extra_payments = {
    "2025-11-10": {"SBI Credit Card": 5000},
    "2025-12-01": {"HDFC Home Loan": 20000},
}

In [7]:
def calculate_emi(principal, annual_rate, months):
    """Calculate EMI using reducing balance formula"""
    if months == 0 or principal == 0:
        return 0
    monthly_rate = annual_rate / 1200  # Convert to monthly decimal
    if monthly_rate == 0:
        return principal / months
    emi = principal * monthly_rate * math.pow(1 + monthly_rate, months) / (math.pow(1 + monthly_rate, months) - 1)
    return round(emi, 2)

def monthly_interest(balance, apr):
    """Calculate one month's interest"""
    return balance * (apr / 1200)


In [8]:
for loan in debts["loans"]:
    loan["outstanding"] = loan["principal"]
    loan["months_paid"] = 0
    loan["emi"] = calculate_emi(loan["principal"], loan["apr"], loan["tenure_months"])
    loan["in_moratorium"] = "moratorium_until" in loan

# Initialize credit cards
for card in debts["credit_cards"]:
    card["outstanding"] = card["balance"]
    card["unbilled_balance"] = card["balance"]  # Current spending
    card["billed_balance"] = 0                   # Statement balance
    card["last_statement_date"] = None
    card["payment_due_date"] = None
    card["in_grace_period"] = True


In [9]:
start_date = datetime(2025, 11, 5)
end_date = datetime(2026, 5, 5)  # 6 months
current_date = start_date

transactions = []
monthly_summaries = []

while current_date <= end_date:
    date_str = current_date.strftime("%Y-%m-%d")
    
    # ===== CREDIT CARDS =====
    for card in debts["credit_cards"]:
        # Generate monthly statement
        if current_date.day == card["billing_day"]:
            card["billed_balance"] = card["unbilled_balance"]
            card["unbilled_balance"] = 0
            card["last_statement_date"] = current_date
            card["payment_due_date"] = current_date + timedelta(days=card["grace_days"]-1)
            card["in_grace_period"] = True
            
            transactions.append({
                "Date": date_str,
                "Debt": card["name"],
                "Event": "Statement Generated",
                "Amount": 0,
                "Balance": card["outstanding"]
            })
        
        # Check payment due
        if current_date == card["payment_due_date"]:
            min_payment = max(card["billed_balance"] * card["min_percent"] / 100, 500)
            min_payment = min(min_payment, card["outstanding"])
            
            # Apply payment (minimum for now)
            card["outstanding"] -= min_payment
            card["billed_balance"] -= min_payment
            
            transactions.append({
                "Date": date_str,
                "Debt": card["name"],
                "Event": "Minimum Payment",
                "Amount": -min_payment,
                "Balance": card["outstanding"]
            })
            
            # If not fully paid, exit grace period
            if card["billed_balance"] > 10:  # Small tolerance
                card["in_grace_period"] = False
        
        # Charge interest if grace period ended
        if not card["in_grace_period"] and card["outstanding"] > 0:
            daily_rate = card["apr"] / 36500
            daily_interest = card["outstanding"] * daily_rate
            card["outstanding"] += daily_interest
            card["unbilled_balance"] += daily_interest
            
            transactions.append({
                "Date": date_str,
                "Debt": card["name"],
                "Event": "Interest",
                "Amount": daily_interest,
                "Balance": card["outstanding"]
            })
    
    # ===== LOANS (EMI on due day of month) =====
    for loan in debts["loans"]:
        if current_date.day == loan["emi_day"] and loan["outstanding"] > 0:
            # Check if in moratorium
            if loan.get("in_moratorium"):
                moratorium_end = datetime.strptime(loan["moratorium_until"], "%Y-%m-%d")
                if current_date < moratorium_end:
                    # Interest-only payment during moratorium
                    interest = monthly_interest(loan["outstanding"], loan["apr"])
                    loan["outstanding"] += interest  # Interest capitalizes
                    
                    transactions.append({
                        "Date": date_str,
                        "Debt": loan["name"],
                        "Event": "Moratorium Interest",
                        "Amount": interest,
                        "Balance": loan["outstanding"]
                    })
                    current_date += timedelta(days=1)
                    continue
                else:
                    loan["in_moratorium"] = False
                    # Recalculate EMI with capitalized interest
                    remaining_months = loan["tenure_months"] - loan["months_paid"]
                    loan["emi"] = calculate_emi(loan["outstanding"], loan["apr"], remaining_months)
            
            # Calculate interest component
            monthly_rate = loan["apr"] / 1200
            interest_component = loan["outstanding"] * monthly_rate
            principal_component = loan["emi"] - interest_component
            
            # Handle last payment
            if principal_component >= loan["outstanding"]:
                principal_component = loan["outstanding"]
                actual_payment = principal_component + interest_component
            else:
                actual_payment = loan["emi"]
            
            loan["outstanding"] -= principal_component
            loan["months_paid"] += 1
            
            transactions.append({
                "Date": date_str,
                "Debt": loan["name"],
                "Event": f"EMI (₹{interest_component:.0f} int + ₹{principal_component:.0f} prin)",
                "Amount": -actual_payment,
                "Balance": loan["outstanding"]
            })
# ===== EXTRA PAYMENTS =====
    if date_str in extra_payments:
        for debt_name, amount in extra_payments[date_str].items():
            # Find the debt
            for card in debts["credit_cards"]:
                if card["name"] == debt_name:
                    card["outstanding"] -= amount
                    if card["billed_balance"] > 0:
                        card["billed_balance"] -= min(amount, card["billed_balance"])
                    transactions.append({
                        "Date": date_str,
                        "Debt": debt_name,
                        "Event": "Extra Payment",
                        "Amount": -amount,
                        "Balance": card["outstanding"]
                    })
            
            for loan in debts["loans"]:
                if loan["name"] == debt_name:
                    loan["outstanding"] -= amount
                    transactions.append({
                        "Date": date_str,
                        "Debt": debt_name,
                        "Event": "Extra Payment (reduces principal)",
                        "Amount": -amount,
                        "Balance": loan["outstanding"]
                    })
    
    current_date += timedelta(days=1)

In [10]:
# ------------------- REPORTING -------------------
df = pd.DataFrame(transactions)
df["Amount"] = df["Amount"].round(2)
df["Balance"] = df["Balance"].round(2)

print("=" * 80)
print("DEBT TRANSACTION DIARY (6 MONTHS)")
print("=" * 80)
print(df.to_string(index=False))

print("\n" + "=" * 80)
print("FINAL BALANCES")
print("=" * 80)
for card in debts["credit_cards"]:
    print(f"{card['name']:25} → ₹{card['outstanding']:>12,.2f}")
for loan in debts["loans"]:
    status = " (PAID OFF! 🎉)" if loan["outstanding"] < 1 else ""
    print(f"{loan['name']:25} → ₹{loan['outstanding']:>12,.2f}{status}")

print("\n" + "=" * 80)
print("INTEREST ANALYSIS")
print("=" * 80)
total_interest = df[df["Event"].str.contains("Interest|Moratorium", na=False)]["Amount"].sum()
total_payments = -df[df["Amount"] < 0]["Amount"].sum()
print(f"Total Interest Paid (6 months): ₹{total_interest:,.2f}")
print(f"Total Payments Made:            ₹{total_payments:,.2f}")
print(f"Principal Reduction:            ₹{total_payments - total_interest:,.2f}")

print("\n" + "=" * 80)
print("EMI DETAILS")
print("=" * 80)
for loan in debts["loans"]:
    if not loan.get("in_moratorium", False):
        print(f"{loan['name']:25} EMI: ₹{loan['emi']:>10,.2f}/month for {loan['tenure_months']} months")
    else:
        print(f"{loan['name']:25} In moratorium until {loan['moratorium_until']}")

DEBT TRANSACTION DIARY (6 MONTHS)
      Date            Debt                             Event    Amount    Balance
2025-11-05  HDFC Home Loan     EMI (₹17708 int + ₹3987 prin) -21695.58 2496012.75
2025-11-10   Axis Car Loan      EMI (₹4750 int + ₹7851 prin) -12601.12  592148.88
2025-11-10 SBI Credit Card                     Extra Payment  -5000.00   45000.00
2025-11-15  Education Loan               Moratorium Interest   3500.00  403500.00
2025-12-01 SBI Credit Card               Statement Generated      0.00   45000.00
2025-12-01  HDFC Home Loan Extra Payment (reduces principal) -20000.00 2476012.75
2025-12-05  HDFC Home Loan     EMI (₹17538 int + ₹4157 prin) -21695.58 2471855.60
2025-12-10   Axis Car Loan      EMI (₹4688 int + ₹7913 prin) -12601.12  584235.61
2025-12-15  Education Loan               Moratorium Interest   3530.63  407030.62
2025-12-18 SBI Credit Card                   Minimum Payment  -2500.00   42500.00
2025-12-18 SBI Credit Card                          Interest    